In [ ]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Рабочий код в ячейке ниже. Он приводит таблицу в плоский вид

In [ ]:
import pandas as pd
import os
def day_card_processing(file_dir, file_name, result_dir):
    df = pd.read_csv(file_dir+file_name)
    
    df_futures = df[df['contract_type'] == 'F']
    df_futures = df_futures.fillna(0)
    df_futures = df_futures.sort_values(by = ['isin','iz_fiz'], ascending = [1,1])

    df_futures = df_futures.drop(['change_prev_week_short_abs', 'change_prev_week_long_abs', 'change_prev_week_short_perc', 'change_prev_week_long_perc', 'Unnamed: 13'], axis = 1)

    df_futures[['clients_in_long_fiz', 'clients_in_short_fiz', 'short_position_fiz', 'long_position_fiz']] = 0
    df_futures[['clients_in_long_jur', 'clients_in_short_jur', 'short_position_jur', 'long_position_jur', 'check']] = 0


    pos_col = [5,6,7,8]
    fiz_col = [9,10,11,12]
    jur_col = [13,14,15,16]

    ind = list(df_futures.index)

    I = len(df_futures)

    wipelist = []

    for i in range(0, I):
        if df_futures.iloc[i,1] == df_futures.iloc[(i-1),1]:
            if (df_futures.iloc[i,4] == 1):
                df_futures.iloc[i, fiz_col] = df_futures.iloc[i,pos_col]
                df_futures.iloc[i,jur_col] = df_futures.iloc[(i-1), pos_col]
            else:
                df_futures.iloc[i,jur_col] = df_futures.iloc[(i), pos_col]
        else:
            if (df_futures.iloc[i,4] == 1):
                df_futures.iloc[i, fiz_col] = df_futures.iloc[i,pos_col]
            else:
                df_futures.iloc[i,jur_col] = df_futures.iloc[(i), pos_col]

        if df_futures.iloc[i,1] == df_futures.iloc[(i-1),1]:
            wipelist.append(ind[i-1])

        if (df_futures.iloc[i,11]+df_futures.iloc[i,15] == df_futures.iloc[i,12]+df_futures.iloc[i,16]):
            df_futures.iloc[i,17] = 'OK'
        else:
            df_futures.iloc[i,17] = 'Err'
    #print(wipelist)
    df_futures = df_futures.drop(wipelist)

    #df_futures = df_futures[(df_futures.iz_fiz == 1)]
    df_futures.to_excel(result_dir+file_name.replace('.csv', '')+'.xlsx', index = False)
print(("\033[31m {}" .format('Done')))

# Обработка всех карточек в папке 'file_dir' и вывод в 'result_dir'

In [ ]:
imput_names = os.listdir(path="data"+'/IN')
file_dir = "data"+'/IN'
#file_name = '/'+'derivatives-open-positions-list-2023627.csv'
result_dir = "data"+'/OUT/Processed'

for i in range(0, len(imput_names)):
    #print(file_dir+'/'+imput_names[i])
    file_name = '/'+imput_names[i]
    print(file_name)
    day_card_processing(file_dir, file_name, result_dir)

print(("\033[31m {}" .format('Done')))

# Объединение результатов в один .xlsx

In [ ]:
path="data"+'/OUT/Processed'
#os.remove((path[:-10])+'/All_days_consolidated.xlsx')
res_names = os.listdir(path)
df_consolidated = pd.DataFrame()
for i in range(0, len(res_names)):
        file_name = '/'+res_names[i]
        df_current = pd.read_excel(path+file_name)
        df_consolidated = pd.concat([df_consolidated, df_current], ignore_index = 1)
        print(file_name)
df_consolidated=df_consolidated.sort_values(by = ['moment','isin'], ascending = [1,1])

df_consolidated = df_consolidated[df_consolidated['long_position_fiz']+df_consolidated['short_position_fiz'] != 0 ]
df_consolidated = df_consolidated[df_consolidated['long_position_jur']+df_consolidated['short_position_jur'] != 0 ]

df_consolidated.insert(loc = 18, column = 'pos_shares', value = '>>>')

df_consolidated.insert(loc = 19, column = 'short_pos_share_fiz', value = (df_consolidated['short_position_fiz'])/(df_consolidated['long_position_fiz'] + df_consolidated['short_position_fiz']))
df_consolidated.insert(loc = 20, column = 'long_pos_share_fiz', value = (df_consolidated['long_position_fiz'])/(df_consolidated['long_position_fiz'] + df_consolidated['short_position_fiz']))

df_consolidated.insert(loc = 21, column = 'short_pos_share_jur', value = (df_consolidated['short_position_jur'])/(df_consolidated['long_position_jur'] + df_consolidated['short_position_jur']))
df_consolidated.insert(loc = 22, column = 'long_pos_share_jur', value = (df_consolidated['long_position_jur'])/(df_consolidated['long_position_jur'] + df_consolidated['short_position_jur']))

df_consolidated.to_csv("data/OUT/All_days_consolidated.csv", index=False)
df_consolidated.to_excel("data/OUT/All_days_consolidated.xlsx", index=False)
print(("\033[31m {}" .format('Done')))